# 🔮 Customer Churn Prediction — End-to-End ML Pipeline

**Author:** Your Name  
**Date:** March 2026  
**Dataset:** Telecom Customer Churn (7,500 customers, 18 features)

---

## 📋 Table of Contents

1. [Project Overview & Business Context](#1)
2. [Data Loading & Initial Exploration](#2)
3. [Exploratory Data Analysis (EDA)](#3)
4. [Statistical Hypothesis Testing](#4)
5. [Feature Engineering](#5)
6. [Model Development & Training](#6)
7. [Model Evaluation & Comparison](#7)
8. [Business Impact Analysis](#8)
9. [Conclusions & Next Steps](#9)

---

<a id="1"></a>
## 1. 🎯 Project Overview & Business Context

### Problem Statement
Customer churn (attrition) is one of the most critical challenges in the telecom industry. Acquiring a new customer costs **5–7x more** than retaining an existing one. This project builds a predictive model to identify customers at high risk of churning, enabling proactive retention strategies.

### Objectives
- Identify the **key drivers** of customer churn through statistical analysis
- Build and compare **multiple ML models** for churn prediction
- Quantify the **business impact** of model-driven retention campaigns
- Provide **actionable recommendations** for reducing churn


<a id="2"></a>
## 2. 📦 Data Loading & Initial Exploration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc,
    precision_recall_curve, average_precision_score, roc_auc_score,
    f1_score, accuracy_score
)
import warnings
warnings.filterwarnings('ignore')

# Plotting config
sns.set_theme(style="whitegrid", font_scale=1.1)
COLORS = {'primary': '#0EA5E9', 'danger': '#EF4444', 'success': '#10B981', 
          'secondary': '#6366F1', 'accent': '#F59E0B'}
palette = list(COLORS.values())

print("✅ Libraries loaded successfully")

In [ ]:
# Load the dataset
df = pd.read_csv('data/telecom_churn.csv')
print(f"Dataset dimensions: {df.shape[0]:,} customers × {df.shape[1]} features")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
df.head(10)

In [ ]:
# Data types and info
print("=" * 60)
print("DATA TYPES SUMMARY")
print("=" * 60)
print(f"\nNumerical features:  {df.select_dtypes(include='number').shape[1]}")
print(f"Categorical features: {df.select_dtypes(include='object').shape[1]}")
print(f"\nMissing values: {df.isnull().sum().sum()}")
print(f"Duplicate rows: {df.duplicated().sum()}")
print("\n" + "=" * 60)
df.describe().round(2)

In [ ]:
# Target variable distribution
churn_counts = df['Churn'].value_counts()
churn_pct = df['Churn'].value_counts(normalize=True) * 100
print("TARGET VARIABLE: Churn")
print("-" * 40)
print(f"Retained (0): {churn_counts[0]:>6,} ({churn_pct[0]:.1f}%)")
print(f"Churned  (1): {churn_counts[1]:>6,} ({churn_pct[1]:.1f}%)")
print(f"\nClass imbalance ratio: {churn_counts[0]/churn_counts[1]:.2f}:1")

<a id="3"></a>
## 3. 🔍 Exploratory Data Analysis (EDA)

### 3.1 Target Variable Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Pie chart
colors_pie = [COLORS['success'], COLORS['danger']]
wedges, texts, autotexts = axes[0].pie(
    churn_counts, labels=['Retained', 'Churned'], colors=colors_pie,
    autopct='%1.1f%%', startangle=90, explode=(0, 0.06),
    textprops={'fontsize': 13, 'fontweight': 'bold'},
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
axes[0].set_title('Customer Churn Distribution', fontweight='bold', fontsize=14)

# Bar chart
sns.countplot(data=df, x='Churn', palette=colors_pie, ax=axes[1], edgecolor='white', linewidth=1.5)
axes[1].set_xticklabels(['Retained', 'Churned'])
axes[1].set_title('Churn Counts', fontweight='bold', fontsize=14)
for p in axes[1].patches:
    axes[1].annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width()/2., p.get_height()),
                     ha='center', va='bottom', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

### 3.2 Churn by Contract Type

Contract type is one of the strongest indicators of churn. Month-to-month customers have significantly lower switching costs.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ct = pd.crosstab(df['Contract'], df['Churn'], normalize='index') * 100
ct.columns = ['Retained %', 'Churned %']
ct.plot(kind='bar', stacked=True, color=[COLORS['success'], COLORS['danger']], ax=ax, edgecolor='white')
ax.set_title('Churn Rate by Contract Type', fontweight='bold', fontsize=14)
ax.set_ylabel('Percentage (%)')
ax.set_xlabel('')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
for c in ax.containers:
    ax.bar_label(c, fmt='%.1f%%', label_type='center', fontsize=10, fontweight='bold', color='white')
plt.tight_layout()
plt.show()

print("\nChurn rates by contract type:")
for contract in df['Contract'].unique():
    rate = df[df['Contract'] == contract]['Churn'].mean() * 100
    n = len(df[df['Contract'] == contract])
    print(f"  {contract:20s}: {rate:5.1f}% churn (n={n:,})")

### 3.3 Tenure & Monthly Charges Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for label, group in df.groupby('Churn'):
    color = COLORS['danger'] if label == 1 else COLORS['success']
    name = 'Churned' if label == 1 else 'Retained'
    axes[0].hist(group['Tenure_Months'], bins=30, alpha=0.6, color=color, label=name, edgecolor='white')
    axes[1].hist(group['MonthlyCharges'], bins=30, alpha=0.6, color=color, label=name, edgecolor='white')
axes[0].set_title('Tenure Distribution by Churn Status', fontweight='bold')
axes[0].set_xlabel('Tenure (Months)')
axes[0].legend()
axes[1].set_title('Monthly Charges by Churn Status', fontweight='bold')
axes[1].set_xlabel('Monthly Charges ($)')
axes[1].legend()
plt.tight_layout()
plt.show()

print("Summary Statistics: Tenure")
print(df.groupby('Churn')['Tenure_Months'].describe().round(1).to_string())
print("\nSummary Statistics: Monthly Charges")
print(df.groupby('Churn')['MonthlyCharges'].describe().round(1).to_string())

### 3.4 Correlation Analysis

In [ ]:
numeric_cols = ['Tenure_Months', 'MonthlyCharges', 'TotalCharges', 
                'NumSupportTickets', 'AvgDataUsageGB', 'NumReferrals', 'SeniorCitizen', 'Churn']
fig, ax = plt.subplots(figsize=(10, 8))
corr = df[numeric_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=1, ax=ax, vmin=-1, vmax=1)
ax.set_title('Feature Correlation Matrix', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

# Top correlations with Churn
churn_corr = corr['Churn'].drop('Churn').abs().sort_values(ascending=False)
print("Top correlations with Churn:")
for feat, val in churn_corr.items():
    direction = "+" if corr.loc[feat, 'Churn'] > 0 else "-"
    print(f"  {feat:25s}: {direction}{val:.3f}")

### 3.5 Churn by Payment Method & Internet Service

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

churn_payment = df.groupby('PaymentMethod')['Churn'].mean().sort_values(ascending=False) * 100
churn_payment.plot(kind='barh', color=[COLORS['danger'] if v > 35 else COLORS['primary'] for v in churn_payment.values], 
                   ax=axes[0], edgecolor='white', linewidth=1.5)
axes[0].set_title('Churn Rate by Payment Method', fontweight='bold')
axes[0].set_xlabel('Churn Rate (%)')
for i, v in enumerate(churn_payment.values):
    axes[0].text(v + 0.5, i, f'{v:.1f}%', va='center', fontweight='bold')

churn_internet = df.groupby('InternetService')['Churn'].mean().sort_values(ascending=False) * 100
churn_internet.plot(kind='barh', color=[COLORS['danger'] if v > 35 else COLORS['primary'] for v in churn_internet.values], 
                    ax=axes[1], edgecolor='white', linewidth=1.5)
axes[1].set_title('Churn Rate by Internet Service', fontweight='bold')
axes[1].set_xlabel('Churn Rate (%)')
for i, v in enumerate(churn_internet.values):
    axes[1].text(v + 0.5, i, f'{v:.1f}%', va='center', fontweight='bold')
plt.tight_layout()
plt.show()

### 3.6 Support Tickets & Churn Relationship

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ticket_churn = df.groupby('NumSupportTickets')['Churn'].agg(['mean', 'count']).reset_index()
ticket_churn.columns = ['Tickets', 'ChurnRate', 'Count']
ticket_churn = ticket_churn[ticket_churn['Count'] >= 20]
ax.bar(ticket_churn['Tickets'], ticket_churn['ChurnRate'] * 100, color=COLORS['primary'], edgecolor='white', alpha=0.85)
z = np.polyfit(ticket_churn['Tickets'], ticket_churn['ChurnRate'] * 100, 2)
p = np.poly1d(z)
x_smooth = np.linspace(ticket_churn['Tickets'].min(), ticket_churn['Tickets'].max(), 100)
ax.plot(x_smooth, p(x_smooth), color=COLORS['danger'], linewidth=2.5, linestyle='--', label='Polynomial Trend')
ax.set_title('Churn Rate by Number of Support Tickets', fontweight='bold', fontsize=14)
ax.set_xlabel('Number of Support Tickets')
ax.set_ylabel('Churn Rate (%)')
ax.legend()
plt.tight_layout()
plt.show()

<a id="4"></a>
## 4. 📐 Statistical Hypothesis Testing

We perform formal statistical tests to validate our EDA observations.

In [ ]:
print("=" * 70)
print("STATISTICAL HYPOTHESIS TESTS (α = 0.05)")
print("=" * 70)

# Test 1: Tenure difference
retained = df[df['Churn'] == 0]['Tenure_Months']
churned = df[df['Churn'] == 1]['Tenure_Months']
t_stat, p_val = stats.ttest_ind(retained, churned)
d = (retained.mean() - churned.mean()) / np.sqrt((retained.std()**2 + churned.std()**2) / 2)
print(f"\n1. Tenure: Retained vs. Churned (Welch's t-test)")
print(f"   Retained mean: {retained.mean():.1f} months | Churned mean: {churned.mean():.1f} months")
print(f"   t = {t_stat:.4f}, p = {p_val:.2e}, Cohen's d = {d:.3f}")
print(f"   → {'SIGNIFICANT' if p_val < 0.05 else 'Not significant'} — {'Large' if abs(d) > 0.8 else 'Medium' if abs(d) > 0.5 else 'Small'} effect size")

# Test 2: Monthly Charges
retained_mc = df[df['Churn'] == 0]['MonthlyCharges']
churned_mc = df[df['Churn'] == 1]['MonthlyCharges']
t2, p2 = stats.ttest_ind(retained_mc, churned_mc)
d2 = (churned_mc.mean() - retained_mc.mean()) / np.sqrt((retained_mc.std()**2 + churned_mc.std()**2) / 2)
print(f"\n2. Monthly Charges: Retained vs. Churned")
print(f"   Retained mean: ${retained_mc.mean():.2f} | Churned mean: ${churned_mc.mean():.2f}")
print(f"   t = {t2:.4f}, p = {p2:.2e}, Cohen's d = {d2:.3f}")
print(f"   → {'SIGNIFICANT' if p2 < 0.05 else 'Not significant'}")

# Test 3: Chi-squared — Contract vs Churn
contingency = pd.crosstab(df['Contract'], df['Churn'])
chi2, p3, dof, expected = stats.chi2_contingency(contingency)
n_total = contingency.sum().sum()
cramers_v = np.sqrt(chi2 / (n_total * (min(contingency.shape) - 1)))
print(f"\n3. Contract Type vs. Churn (Chi-squared test)")
print(f"   χ² = {chi2:.2f}, p = {p3:.2e}, df = {dof}, Cramér's V = {cramers_v:.3f}")
print(f"   → {'SIGNIFICANT' if p3 < 0.05 else 'Not significant'} — {'Strong' if cramers_v > 0.3 else 'Moderate' if cramers_v > 0.1 else 'Weak'} association")

# Test 4: Chi-squared — Payment Method vs Churn
contingency2 = pd.crosstab(df['PaymentMethod'], df['Churn'])
chi2_2, p4, dof2, _ = stats.chi2_contingency(contingency2)
cramers_v2 = np.sqrt(chi2_2 / (n_total * (min(contingency2.shape) - 1)))
print(f"\n4. Payment Method vs. Churn (Chi-squared test)")
print(f"   χ² = {chi2_2:.2f}, p = {p4:.2e}, df = {dof2}, Cramér's V = {cramers_v2:.3f}")
print(f"   → {'SIGNIFICANT' if p4 < 0.05 else 'Not significant'}")

# Test 5: Support Tickets — Mann-Whitney U
u_stat, p5 = stats.mannwhitneyu(
    df[df['Churn']==1]['NumSupportTickets'],
    df[df['Churn']==0]['NumSupportTickets'],
    alternative='greater'
)
print(f"\n5. Support Tickets: Churned > Retained (Mann-Whitney U)")
print(f"   U = {u_stat:.0f}, p = {p5:.2e}")
print(f"   → {'SIGNIFICANT' if p5 < 0.05 else 'Not significant'}")
print("\n" + "=" * 70)

<a id="5"></a>
## 5. ⚙️ Feature Engineering

In [ ]:
# Create engineered features
df_model = df.drop('CustomerID', axis=1).copy()

# Encode categoricals
cat_cols = df_model.select_dtypes(include='object').columns
le_dict = {}
for col in cat_cols:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col])
    le_dict[col] = le

# New features
df_model['ChargesPerMonth_Ratio'] = df_model['TotalCharges'] / (df_model['Tenure_Months'] + 1)
df_model['TicketsPerTenure'] = df_model['NumSupportTickets'] / (df_model['Tenure_Months'] + 1)
df_model['HighValue'] = (df_model['MonthlyCharges'] > df_model['MonthlyCharges'].quantile(0.75)).astype(int)
df_model['NewCustomer'] = (df_model['Tenure_Months'] <= 6).astype(int)

print("Engineered Features:")
print("-" * 50)
print("• ChargesPerMonth_Ratio — Total charges normalized by tenure")
print("• TicketsPerTenure — Support ticket frequency relative to tenure")
print("• HighValue — Binary flag for top-25% monthly spenders")
print("• NewCustomer — Binary flag for customers with ≤ 6 months tenure")
print(f"\nFinal feature matrix: {df_model.shape[0]:,} rows × {df_model.shape[1]-1} features")
df_model.head()

In [ ]:
# Train/test split
X = df_model.drop('Churn', axis=1)
y = df_model['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {X_train.shape[0]:,} samples")
print(f"Test set:     {X_test.shape[0]:,} samples")
print(f"\nTrain churn rate: {y_train.mean():.2%}")
print(f"Test churn rate:  {y_test.mean():.2%}")

<a id="6"></a>
## 6. 🤖 Model Development & Training

We train and compare three models of increasing complexity:
1. **Logistic Regression** — interpretable baseline
2. **Random Forest** — ensemble of decision trees
3. **Gradient Boosting** — sequential boosted ensemble

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, C=1.0, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=12, min_samples_leaf=5, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42),
}

results = {}
for name, model in models.items():
    print(f"\n{'='*50}")
    print(f"Training: {name}")
    print(f"{'='*50}")
    
    if name == 'Logistic Regression':
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
        cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='roc_auc')
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]
        cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='roc_auc')
    
    results[name] = {
        'model': model, 'y_pred': y_pred, 'y_proba': y_proba,
        'accuracy': accuracy_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'auc': roc_auc_score(y_test, y_proba),
        'cv_mean': cv_scores.mean(), 'cv_std': cv_scores.std(),
    }
    
    print(f"  Accuracy:  {results[name]['accuracy']:.4f}")
    print(f"  F1-Score:  {results[name]['f1']:.4f}")
    print(f"  AUC-ROC:   {results[name]['auc']:.4f}")
    print(f"  CV AUC:    {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
    print(f"\n{classification_report(y_test, y_pred, target_names=['Retained', 'Churned'])}")

<a id="7"></a>
## 7. 📊 Model Evaluation & Comparison

### 7.1 ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
for i, (name, res) in enumerate(results.items()):
    fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
    ax.plot(fpr, tpr, label=f"{name} (AUC={res['auc']:.3f})", linewidth=2.5, color=palette[i])
ax.plot([0, 1], [0, 1], 'k--', alpha=0.4)
ax.fill_between([0, 1], [0, 1], alpha=0.03, color='gray')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Model Comparison', fontweight='bold', fontsize=14)
ax.legend(loc='lower right', fontsize=11, frameon=True, fancybox=True)
plt.tight_layout()
plt.show()

### 7.2 Precision-Recall Curves

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
for i, (name, res) in enumerate(results.items()):
    precision, recall, _ = precision_recall_curve(y_test, res['y_proba'])
    ap = average_precision_score(y_test, res['y_proba'])
    ax.plot(recall, precision, label=f"{name} (AP={ap:.3f})", linewidth=2.5, color=palette[i])
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curves', fontweight='bold', fontsize=14)
ax.legend(loc='upper right', fontsize=11, frameon=True)
plt.tight_layout()
plt.show()

### 7.3 Confusion Matrix — Best Model

In [ ]:
best_name = max(results, key=lambda k: results[k]['auc'])
best = results[best_name]
fig, ax = plt.subplots(figsize=(7, 6))
cm = confusion_matrix(y_test, best['y_pred'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Retained', 'Churned'], yticklabels=['Retained', 'Churned'],
            linewidths=2, linecolor='white', annot_kws={'size': 16, 'fontweight': 'bold'})
ax.set_title(f'Confusion Matrix — {best_name}', fontweight='bold', fontsize=14)
ax.set_ylabel('Actual')
ax.set_xlabel('Predicted')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True Negatives:  {tn:,}  |  False Positives: {fp:,}")
print(f"False Negatives: {fn:,}  |  True Positives:  {tp:,}")
print(f"\nPrecision: {tp/(tp+fp):.3f}  |  Recall: {tp/(tp+fn):.3f}")

### 7.4 Feature Importance

In [ ]:
gb_model = results['Gradient Boosting']['model']
feat_imp = pd.Series(gb_model.feature_importances_, index=X.columns).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(10, 8))
feat_imp.tail(15).plot(kind='barh', color=COLORS['primary'], ax=ax, edgecolor='white')
ax.set_title('Top 15 Feature Importances — Gradient Boosting', fontweight='bold', fontsize=14)
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

print("\nTop 10 Most Important Features:")
for i, (feat, imp) in enumerate(feat_imp.tail(10).iloc[::-1].items(), 1):
    print(f"  {i:2d}. {feat:30s}: {imp:.4f}")

### 7.5 Model Comparison Summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
metrics_list = ['accuracy', 'f1', 'auc']
titles = ['Accuracy', 'F1-Score', 'AUC-ROC']
for idx, (metric, title) in enumerate(zip(metrics_list, titles)):
    vals = [results[m][metric] for m in results]
    bars = axes[idx].bar(results.keys(), vals, color=palette[:3], edgecolor='white')
    axes[idx].set_title(title, fontweight='bold')
    axes[idx].set_ylim(min(vals) - 0.05, min(max(vals) + 0.03, 1.0))
    axes[idx].set_xticklabels(results.keys(), rotation=15, ha='right')
    for bar, v in zip(bars, vals):
        axes[idx].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                       f'{v:.3f}', ha='center', fontweight='bold', fontsize=11)
plt.tight_layout()
plt.show()

# Summary table
summary_df = pd.DataFrame({
    'Model': list(results.keys()),
    'Accuracy': [r['accuracy'] for r in results.values()],
    'F1-Score': [r['f1'] for r in results.values()],
    'AUC-ROC': [r['auc'] for r in results.values()],
    'CV AUC (mean±std)': [f"{r['cv_mean']:.4f}±{r['cv_std']:.4f}" for r in results.values()],
}).set_index('Model')
summary_df

### 7.6 Learning Curve

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
train_sizes, train_scores, val_scores = learning_curve(
    GradientBoostingClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42),
    X_train, y_train, cv=5, scoring='roc_auc', train_sizes=np.linspace(0.1, 1.0, 10), n_jobs=-1
)
ax.plot(train_sizes, train_scores.mean(axis=1), 'o-', color=COLORS['primary'], label='Training', linewidth=2)
ax.fill_between(train_sizes, train_scores.mean(axis=1) - train_scores.std(axis=1),
                train_scores.mean(axis=1) + train_scores.std(axis=1), alpha=0.15, color=COLORS['primary'])
ax.plot(train_sizes, val_scores.mean(axis=1), 'o-', color=COLORS['danger'], label='Validation', linewidth=2)
ax.fill_between(train_sizes, val_scores.mean(axis=1) - val_scores.std(axis=1),
                val_scores.mean(axis=1) + val_scores.std(axis=1), alpha=0.15, color=COLORS['danger'])
ax.set_title('Learning Curve — Gradient Boosting', fontweight='bold', fontsize=14)
ax.set_xlabel('Training Set Size')
ax.set_ylabel('AUC-ROC')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

<a id="8"></a>
## 8. 💰 Business Impact Analysis

In [ ]:
# Business impact simulation
avg_customer_ltv = 2800  # Average lifetime value
retention_cost = 150     # Cost of retention campaign per customer
success_rate = 0.35      # Estimated success rate of retention intervention

# Using best model predictions
high_risk = (best['y_proba'] >= 0.6).sum()
medium_risk = ((best['y_proba'] >= 0.4) & (best['y_proba'] < 0.6)).sum()
low_risk = (best['y_proba'] < 0.4).sum()

saved_revenue_high = high_risk * success_rate * avg_customer_ltv
campaign_cost_high = high_risk * retention_cost
net_benefit_high = saved_revenue_high - campaign_cost_high

print("=" * 60)
print("BUSINESS IMPACT ANALYSIS")
print("=" * 60)
print(f"\n📊 Risk Segmentation (Test Set: {len(y_test):,} customers)")
print(f"   High Risk   (≥60%): {high_risk:>5,} customers")
print(f"   Medium Risk (40-60%): {medium_risk:>5,} customers")
print(f"   Low Risk    (<40%): {low_risk:>5,} customers")
print(f"\n💰 Financial Projections (High-Risk Targeting)")
print(f"   Avg Customer LTV:     ${avg_customer_ltv:>10,}")
print(f"   Retention Cost/Customer: ${retention_cost:>8,}")
print(f"   Intervention Success Rate: {success_rate:.0%}")
print(f"\n   Estimated Saved Revenue: ${saved_revenue_high:>12,.0f}")
print(f"   Campaign Cost:           ${campaign_cost_high:>12,.0f}")
print(f"   ─────────────────────────────────────────")
print(f"   Net Benefit:             ${net_benefit_high:>12,.0f}")
print(f"\n   ROI: {(net_benefit_high / campaign_cost_high) * 100:.0f}%")

# Annualized (scale to full customer base)
scale = df.shape[0] / len(y_test)
annual_benefit = net_benefit_high * scale * 4  # quarterly interventions
print(f"\n📈 Projected Annual Benefit (full base, quarterly): ${annual_benefit:,.0f}")
print("=" * 60)

<a id="9"></a>
## 9. ✅ Conclusions & Next Steps

### Key Findings

| Finding | Detail |
|:--------|:-------|
| **Strongest churn predictor** | Contract type (month-to-month customers churn 2-3× more) |
| **Payment risk** | Electronic check users have the highest churn rate |
| **Tenure effect** | New customers (≤6 months) are most vulnerable |
| **Support tickets** | Higher ticket count correlates with increased churn |
| **Best model** | Logistic Regression achieved the highest AUC-ROC |

### Recommended Strategies

1. **Contract Migration Incentives** — Offer discounts for month-to-month customers who switch to annual contracts
2. **Early Intervention Program** — Trigger retention campaigns for customers in their first 6 months
3. **Payment Method Optimization** — Incentivize migration from electronic check to auto-pay
4. **Proactive Support** — Flag accounts with ≥4 support tickets for dedicated account management
5. **High-Value Protection** — Dedicated retention team for customers in the top revenue quartile

### Future Work

- **Deep Learning** — Experiment with neural networks and attention-based architectures
- **Survival Analysis** — Model time-to-churn rather than binary churn
- **Causal Inference** — Use uplift modeling to estimate individual treatment effects
- **Real-Time Scoring** — Deploy model as a microservice with <100ms latency
- **Feature Store** — Build a centralized feature store for real-time feature engineering

---
*This analysis was conducted as part of a data science portfolio project. All data is synthetic.*
